In [140]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [141]:
train_df = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/sample_submission.csv")

In [142]:
train_df

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [143]:
test_df

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4272,9266_02,Earth,True,G/1496/S,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,0.0,0.0,Jeron Peter
4273,9269_01,Earth,False,NaN,TRAPPIST-1e,42.0,False,0.0,847.0,17.0,10.0,144.0,Matty Scheron
4274,9271_01,Mars,True,D/296/P,55 Cancri e,NaN,False,0.0,0.0,0.0,0.0,0.0,Jayrin Pore
4275,9273_01,Europa,False,D/297/P,NaN,NaN,False,0.0,2680.0,0.0,0.0,523.0,Kitakan Conale


In [144]:
train_df.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [145]:
train_df["HomePlanet"].value_counts()

HomePlanet
Earth     4602
Europa    2131
Mars      1759
Name: count, dtype: int64

In [146]:
train_df["Destination"].value_counts()

Destination
TRAPPIST-1e      5915
55 Cancri e      1800
PSO J318.5-22     796
Name: count, dtype: int64

In [147]:
train_df["Name"].value_counts().sort_values(ascending=False)

Name
Keitha Josey          2
Gwendy Sykess         2
Ankalik Nateansive    2
Cuses Pread           2
Glenna Valezaley      2
                     ..
Altark Susent         1
Sterion Dianket       1
Janney Hoffergess     1
Alark Eguing          1
Stendy Connelson      1
Name: count, Length: 8473, dtype: int64

In [148]:
from sklearn.model_selection import StratifiedKFold

In [149]:
def target_encode(col, target, df_train, df_test=None):
    skf = StratifiedKFold(n_splits=5)
    encoded_train = df_train[col].copy()
    
    for tr_idx, val_idx in skf.split(df_train, df_train[target]):
        tr_mean = df_train.loc[tr_idx, target].groupby(df_train.loc[tr_idx, col]).mean()
        encoded_train.iloc[val_idx] = df_train.loc[val_idx, col].map(tr_mean)
    
    global_mean = df_train[target].groupby(df_train[col]).mean()
    if df_test is not None:
        encoded_test = df_test[col].map(global_mean).fillna(encoded_train.mean())
        return encoded_train.fillna(encoded_train.mean()), encoded_test
    return encoded_train.fillna(encoded_train.mean())

In [150]:
def change_features(df):
    bool_features = ['CryoSleep', 'VIP']

    for cols in bool_features:
        df[cols] = df[cols].map({True: 1, False: 0}).fillna(0).astype(int)

    df["HomePlanet"] = df["HomePlanet"].map({"Earth": 1, "Europa": 2, "Mars": 3}).fillna(0).astype(int)

    df["Deck"] = df["Cabin"].str.split("/").str[0].fillna("Missing")
    df["Num"] = pd.to_numeric(df["Cabin"].str.split("/").str[1], errors='coerce').fillna(-1).astype(int)
    df["Side"] = df["Cabin"].str.split("/").str[2].fillna("Missing")
    df.drop("Cabin", axis=1, inplace=True)

    for col in ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']:
        df[col] = df[col].fillna(df[col].median())
    
    return df

In [151]:
train_df = change_features(train_df)
test_df = change_features(test_df)

for col in ['HomePlanet', 'Destination', 'Deck', 'Side']:
    train_df[f'{col}_Target'], test_df[f'{col}_Target'] = target_encode(col, 'Transported', train_df, test_df)

train_df['Total_expense'] = train_df[['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']].sum(axis=1)
test_df['Total_expense'] = test_df[['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']].sum(axis=1)

/tmp/ipykernel_159/766069622.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.65109403 0.42911358 0.65109403 ... 0.65109403 0.65109403 0.65109403]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  encoded_train.iloc[val_idx] = df_train.loc[val_idx, col].map(tr_mean)
/tmp/ipykernel_159/766069622.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return encoded_train.fillna(encoded_train.mean()), encoded_test
/tmp/ipykernel_159/766069622.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavi

In [152]:
features_cols = [
    "CryoSleep", "Age", "VIP",
    "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck",
    "HomePlanet_Target", "Destination_Target", "Deck_Target", "Side_Target",
    "Num", "Total_expense"
]

print(train_df.columns)
print(test_df.columns)

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP',
       'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name',
       'Transported', 'Deck', 'Num', 'Side', 'HomePlanet_Target',
       'Destination_Target', 'Deck_Target', 'Side_Target', 'Total_expense'],
      dtype='object')
Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP',
       'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name',
       'Deck', 'Num', 'Side', 'HomePlanet_Target', 'Destination_Target',
       'Deck_Target', 'Side_Target', 'Total_expense'],
      dtype='object')


In [153]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

In [154]:
X = train_df[features_cols]
y = train_df["Transported"].astype(int)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        objective="binary",
        metric="binary_logloss",
        max_depth=8,
        n_estimators=1000,
        learning_rate=0.05,
        random_state=42,
        verbosity=-1,
        nem_leaves=32
    )

    model.fit(X_tr, y_tr)
    val_pred = model.predict(X_val)
    score = accuracy_score(y_val, val_pred)
    scores.append(score)
    print(f"Fold {fold} accuracy: {score}")

print(f"CV accuracy: {np.mean(scores):.4f}")

Fold 0 accuracy: 0.8108108108108109
Fold 1 accuracy: 0.8108108108108109
Fold 2 accuracy: 0.8062104657849338
Fold 3 accuracy: 0.8199079401611047
Fold 4 accuracy: 0.7991944764096662
CV accuracy: 0.8094


In [156]:
X = train_df[features_cols]
y = train_df["Transported"].astype(int)
X_test = test_df[features_cols]

final_model = lgb.LGBMClassifier(
    objective="binary",
    metric="binary_logloss",
    max_depth=8,
    n_estimators=1000,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1,
    nem_leaves=32
)

final_model.fit(X, y)
test_pred = final_model.predict(X_test)
submission = pd.DataFrame({"PassengerId": test_df["PassengerId"], "Transported": test_pred.astype(bool)})
submission.to_csv("submission.csv", index=False)